In [ ]:
import datetime
import time
from pathlib import Path

import pandas as pd
from selenium import webdriver
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

REGULATOR_NAME = 'NL AFM'
now = datetime.datetime.now()
processdate = now.strftime('%Y-%m-%d')
filename = f"{REGULATOR_NAME} SQL Ready {str(now).replace(':', '.')[:-7]}.xlsx"
scriptfolder = Path.cwd()
tempfolder = scriptfolder / 'tempfolder'
tempfolder.mkdir(exist_ok=True)
HEADLESS = False

LIST_SPECS = [
    {
        'list_key': 'NL AFM 1',
        'list_code': '1',
        'page_url': 'https://www.afm.nl/en/sector/registers/vergunningenregisters/beleggingsinstellingen',
        'selector': "a[href*='883bcff1-0f26-442f-9faf-a39ff911b109'][href*='format=csv']",
        'expected_exts': ['.csv'],
        'final_name': 'nl_afm_1_collective_investment_schemes.csv',
        'kind': 'csv',
    },
    {
        'list_key': 'NL AFM 2',
        'list_code': '2',
        'page_url': 'https://www.afm.nl/en/sector/registers/vergunningenregisters/beleggingsinstellingen',
        'selector': "a[href*='register-aifm.xlsx']",
        'expected_exts': ['.xlsx', '.xls'],
        'final_name': 'nl_afm_2_aifmd_licensed_aifms.xlsx',
        'kind': 'aifm',
    },
    {
        'list_key': 'NL AFM 3',
        'list_code': '3',
        'page_url': 'https://www.afm.nl/en/sector/registers/vergunningenregisters/beleggingsinstellingen',
        'selector': "a[href*='register-aifmd-light.xlsx']",
        'expected_exts': ['.xlsx', '.xls'],
        'final_name': 'nl_afm_3_light_aifms.xlsx',
        'kind': 'light',
    },
    {
        'list_key': 'NL AFM 4',
        'list_code': '4',
        'page_url': 'https://www.afm.nl/en/sector/registers/vergunningenregisters/beleggingsondernemingen',
        'selector': "a[href*='8f59acf7-047b-4009-9fa7-90a264e6f3ef'][href*='format=csv']",
        'expected_exts': ['.csv'],
        'final_name': 'nl_afm_4_investment_firms.csv',
        'kind': 'csv',
    },
    {
        'list_key': 'NL AFM 5',
        'list_code': '5',
        'page_url': 'https://www.afm.nl/en/sector/registers/vergunningenregisters/financiele-dienstverleners',
        'selector': "a[href*='04efad81-e254-40fa-8728-94d90447ad4b'][href*='format=csv']",
        'expected_exts': ['.csv'],
        'final_name': 'nl_afm_5_financial_service_providers.csv',
        'kind': 'csv',
    },
]

SQL_COLUMNS = [
    'bvdid', 'priority', 'ListLabel', 'Typology', 'EntryType', 'Name', 'InternalID_1', 'InternalID_1_type',
    'InternalID_2', 'InternalID_2_type', 'InternalID_3', 'InternalID_3_type', 'CoType', 'License_Type',
    'Address_1', 'Address_2', 'City', 'Zip', 'Cntry', 'Phone', 'Fax', 'Website', 'Email', 'RegulationType',
    'RegulationTypeCode', 'RegulationDate', 'CancellationDate', 'RegCtry', 'RegCode', 'ListCode',
    'ListLanguage', 'ListValidityDate', 'ListName', 'ListProcessDate', 'LEI Code', 'BIC SWIFT Code',
    'Name - Mother Company', 'Address_1 - Mother company', 'Address_2 -  Mother company',
    'City - Mother company', 'Zip - Mother company', 'Cntry - Mother company', 'Phone - Mother company', 'Check'
]

def new_sqldict():
    return {column: [] for column in SQL_COLUMNS}

def clear_tempfolder(folder):
    for path in folder.iterdir():
        if path.is_file():
            path.unlink()

def launch_driver(download_dir, headless=HEADLESS):
    options = Options()
    prefs = {
        'plugins.always_open_pdf_externally': True,
        'download.prompt_for_download': False,
        'download.default_directory': str(download_dir.resolve()),
        'download.directory_upgrade': True,
        'profile.default_content_setting_values.automatic_downloads': 1,
        'safebrowsing.enabled': True,
    }
    options.add_experimental_option('prefs', prefs)
    options.add_argument('--disable-search-engine-choice-screen')
    options.add_argument('--window-size=1600,1200')
    if headless:
        options.add_argument('--headless=new')
    driver = webdriver.Chrome(options=options)
    try:
        driver.execute_cdp_cmd('Page.setDownloadBehavior', {'behavior': 'allow', 'downloadPath': str(download_dir.resolve())})
    except Exception:
        pass
    return driver

def deny_cookies(driver):
    for label in ['Deny', 'Reject', 'Decline']:
        try:
            button = WebDriverWait(driver, 3).until(
                EC.element_to_be_clickable((By.XPATH, f"//button[normalize-space()='{label}']"))
            )
            driver.execute_script('arguments[0].click();', button)
            time.sleep(0.5)
            return True
        except TimeoutException:
            pass
        except Exception:
            pass
    return False

def wait_for_download(folder, before_names, expected_exts, final_name, timeout=120):
    end_time = time.time() + timeout
    last_size = None
    stable_checks = 0
    candidate = None
    expected_exts = {ext.lower() for ext in expected_exts}

    while time.time() < end_time:
        new_files = [path for path in folder.iterdir() if path.is_file() and path.name not in before_names]
        complete_files = [path for path in new_files if path.suffix.lower() in expected_exts]
        tmp_files = [path for path in new_files if path.suffix.lower() == '.tmp']

        if complete_files:
            candidate = max(complete_files, key=lambda path: path.stat().st_mtime)
        elif tmp_files:
            candidate = max(tmp_files, key=lambda path: path.stat().st_mtime)

        if candidate and candidate.exists():
            size = candidate.stat().st_size
            if size > 0 and size == last_size:
                stable_checks += 1
            else:
                stable_checks = 0
                last_size = size

            if stable_checks >= 2:
                target = folder / final_name
                if target.exists():
                    target.unlink()
                if candidate != target:
                    candidate.replace(target)
                return target

        time.sleep(1)

    raise TimeoutError(f"Download did not complete: {final_name}")

def download_from_page(driver, spec, folder):
    clear_tempfolder(folder)
    before_names = {path.name for path in folder.iterdir() if path.is_file()}
    driver.get(spec['page_url'])
    time.sleep(2)
    deny_cookies(driver)
    link = WebDriverWait(driver, 30).until(EC.element_to_be_clickable((By.CSS_SELECTOR, spec['selector'])))
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", link)
    try:
        driver.execute_script("arguments[0].removeAttribute('target');", link)
    except Exception:
        pass
    time.sleep(0.5)
    driver.execute_script('arguments[0].click();', link)
    return wait_for_download(folder, before_names, spec['expected_exts'], spec['final_name'])

def read_afm_csv(csv_path):
    dataframe = None
    for encoding in ['cp1252', 'utf-8-sig', 'latin1']:
        try:
            dataframe = pd.read_csv(csv_path, sep=';', dtype=str, encoding=encoding).fillna('')
            break
        except Exception:
            pass
    if dataframe is None:
        raise ValueError(f'Unable to parse CSV file: {csv_path.name}')

    first_col = dataframe.columns[0]
    footer_mask = dataframe[first_col].astype(str).str.startswith(('Disclaimer AFM CSV', 'Date last update'), na=False)
    dataframe = dataframe.loc[~footer_mask].copy()
    dataframe.columns = [str(column).strip() for column in dataframe.columns]
    return dataframe.reset_index(drop=True)

def normalize_header(value):
    return ' '.join(str(value).replace('\xa0', ' ').split()).strip().lower()

def read_afm_excel_table(excel_path):
    raw = pd.read_excel(excel_path, header=None, dtype=str).fillna('')
    counts = raw.apply(lambda row: (row.astype(str).str.strip() != '').sum(), axis=1)
    header_idx = int(counts.idxmax())
    header = [str(value).strip() for value in raw.iloc[header_idx].tolist()]
    dataframe = raw.iloc[header_idx + 1:].copy()
    dataframe.columns = header
    dataframe = dataframe.loc[:, [str(column).strip() != '' for column in dataframe.columns]]
    dataframe = dataframe.replace(r'^\s*$', pd.NA, regex=True).dropna(how='all').fillna('')
    dataframe.columns = [str(column).strip() for column in dataframe.columns]
    return dataframe.reset_index(drop=True)

def get_column(dataframe, *candidates):
    normalized = {normalize_header(column): column for column in dataframe.columns}
    for candidate in candidates:
        match = normalized.get(normalize_header(candidate))
        if match:
            return match
    for candidate in candidates:
        candidate_key = normalize_header(candidate)
        for key, original in normalized.items():
            if candidate_key and candidate_key in key:
                return original
    return None

def append_common_metadata(sqldict, spec):
    sqldict['ListProcessDate'].append(processdate)
    sqldict['RegulationType'].append('Licensed')
    sqldict['RegCtry'].append('NL')
    sqldict['RegCode'].append('AFM')
    sqldict['ListCode'].append(spec['list_code'])

def append_csv_rows(sqldict, dataframe, spec):
    name_col = get_column(dataframe, 'Trade name', 'Handelsnaam', 'Statutory name', 'Statutaire naam')
    city_col = get_column(dataframe, 'Place of residence', 'Plaats')
    country_col = get_column(dataframe, 'Country', 'Land')

    for _, row in dataframe.iterrows():
        name = str(row.get(name_col, '')).strip() if name_col else ''
        if not name:
            continue
        sqldict['Name'].append(name)
        sqldict['City'].append(str(row.get(city_col, '')).strip() if city_col else '')
        sqldict['Cntry'].append(str(row.get(country_col, '')).strip() if country_col else '')
        append_common_metadata(sqldict, spec)

def append_aifm_rows(sqldict, dataframe, spec):
    name_col = get_column(dataframe, 'Naam Beleggingsinstelling')
    vergunning_col = get_column(dataframe, 'Vergunningnummer')
    fund_id_col = get_column(dataframe, 'AFM Fonds ID')

    for _, row in dataframe.iterrows():
        name = str(row.get(name_col, '')).strip() if name_col else ''
        if not name:
            continue
        sqldict['Name'].append(name)
        sqldict['InternalID_1'].append(str(row.get(vergunning_col, '')).strip() if vergunning_col else '')
        sqldict['InternalID_1_type'].append('Vergunningnummer')
        sqldict['InternalID_2'].append(str(row.get(fund_id_col, '')).strip() if fund_id_col else '')
        sqldict['InternalID_2_type'].append('AFM Fonds ID')
        append_common_metadata(sqldict, spec)

def append_light_rows(sqldict, dataframe, spec):
    management_col = get_column(dataframe, 'Name of management company', 'Naam beheerder', 'Naam beheermaatschappij')
    scheme_col = get_column(dataframe, 'Name of collective investment scheme', 'Naam beleggingsinstelling', 'Naam fonds')
    fund_id_col = get_column(dataframe, 'AFM Fonds ID')

    for _, row in dataframe.iterrows():
        manager = str(row.get(management_col, '')).strip() if management_col else ''
        scheme = str(row.get(scheme_col, '')).strip() if scheme_col else ''
        name = ' - '.join(part for part in [manager, scheme] if part)
        if not name:
            continue
        sqldict['Name'].append(name)
        sqldict['InternalID_2'].append(str(row.get(fund_id_col, '')).strip() if fund_id_col else '')
        sqldict['InternalID_2_type'].append('AFM Fonds ID')
        append_common_metadata(sqldict, spec)

def pad_sqldict(sqldict):
    max_len = max(len(values) for values in sqldict.values()) if sqldict else 0
    for key, values in sqldict.items():
        if len(values) < max_len:
            values.extend([''] * (max_len - len(values)))
    return sqldict

def collect_sql_ready(headless=HEADLESS):
    sqldict = new_sqldict()
    counts = []
    driver = launch_driver(tempfolder, headless=headless)
    try:
        for index, spec in enumerate(LIST_SPECS, start=1):
            print(f"[INFO] Working {index}/{len(LIST_SPECS)} ({spec['list_key']})")
            downloaded_file = download_from_page(driver, spec, tempfolder)
            if spec['kind'] == 'csv':
                dataframe = read_afm_csv(downloaded_file)
                append_csv_rows(sqldict, dataframe, spec)
            elif spec['kind'] == 'aifm':
                dataframe = read_afm_excel_table(downloaded_file)
                append_aifm_rows(sqldict, dataframe, spec)
            else:
                dataframe = read_afm_excel_table(downloaded_file)
                append_light_rows(sqldict, dataframe, spec)
            counts.append({'ListCode': spec['list_code'], 'Rows': len(dataframe), 'File': downloaded_file.name})
            pad_sqldict(sqldict)
    finally:
        driver.quit()

    return pd.DataFrame(pad_sqldict(sqldict)), pd.DataFrame(counts)

print(f"Running {REGULATOR_NAME} Web Scraping Tool v.2.0")
sql_ready_df, collection_summary = collect_sql_ready(headless=HEADLESS)
output_path = scriptfolder / filename
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    sql_ready_df.to_excel(writer, sheet_name='SQL Ready', index=False)
print(f"Saved SQL Ready output to: {output_path}")
collection_summary
